# IMDb Sentiment Analysis with a Bidirectional LSTM

Binary sentiment classification (positive / negative) of IMDb movie reviews.

**Pipeline:** load → explore → pad → build → train with early stopping → evaluate once on the held-out test set → predict on new text.

**Evaluation protocol**
- Validation uses 20% of the training data; the 25,000-review test set is evaluated exactly once, at the end.
- Early stopping monitors validation loss and restores the best epoch's weights.
- Padding zeros are masked, and long reviews are truncated from the start so their endings are kept.

## 1. Setup

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
VOCAB_SIZE = 10_000   # keep the 10,000 most frequent words
MAX_LEN = 250         # tokens per review after padding / truncation
EMBED_DIM = 32
LSTM_UNITS = 64
BATCH_SIZE = 32
MAX_EPOCHS = 10
VAL_SPLIT = 0.2

# Reserved indices in the Keras IMDb encoding; real words start at INDEX_FROM.
PAD, START, OOV, INDEX_FROM = 0, 1, 2, 3

tf.keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))

## 2. Load the data

In [ ]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print(f"Training reviews: {len(X_train):,}")
print(f"Test reviews:     {len(X_test):,}")

## 3. Explore the data

In [ ]:
word_index = imdb.get_word_index()
index_word = {i + INDEX_FROM: w for w, i in word_index.items()}
index_word.update({PAD: "<PAD>", START: "<START>", OOV: "<UNK>"})

def decode(sequence, n=40):
    return " ".join(index_word.get(int(i), "?") for i in sequence[:n])

print("Token ids:", [int(i) for i in X_train[0][:12]])
print("Decoded:  ", decode(X_train[0]))
print("Label:    ", "positive" if y_train[0] == 1 else "negative")

In [ ]:
lengths = np.array([len(r) for r in X_train])

print(f"Mean length:   {lengths.mean():.0f} tokens")
print(f"Median length: {np.median(lengths):.0f} tokens")
print(f"Longest:       {lengths.max():,} tokens")
print(f"Reviews <= {MAX_LEN} tokens: {(lengths <= MAX_LEN).mean():.1%}")
print(f"Class balance: {int(y_train.sum()):,} positive / {int(len(y_train) - y_train.sum()):,} negative")

plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=60, range=(0, 1200), color="steelblue")
plt.axvline(MAX_LEN, color="red", linestyle="--", label=f"MAX_LEN = {MAX_LEN}")
plt.title("Review length distribution")
plt.xlabel("Tokens per review")
plt.ylabel("Reviews")
plt.legend()
plt.show()

## 4. Pad and truncate

In [ ]:
# Pad at the end (masked in the model); truncate from the start so the review's ending is kept.
X_train = pad_sequences(X_train, maxlen=MAX_LEN, padding="post", truncating="pre")
X_test = pad_sequences(X_test, maxlen=MAX_LEN, padding="post", truncating="pre")

print("Training tensor:", X_train.shape)
print("Test tensor:    ", X_test.shape)

## 5. Build the model

In [ ]:
model = tf.keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(VOCAB_SIZE, EMBED_DIM, mask_zero=True),   # index 0 = padding, ignored downstream
    layers.Bidirectional(layers.LSTM(LSTM_UNITS)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## 6. Train

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,   # last 20% of the training data; the test set is not used here
    callbacks=[early_stop],
)

best_epoch = int(np.argmin(history.history["val_loss"])) + 1
print(f"Stopped after {len(history.history['val_loss'])} epochs; best epoch: {best_epoch}")

## 7. Evaluate on the held-out test set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss:     {test_loss:.4f}")

In [ ]:
fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric, title in [(ax_acc, "accuracy", "Accuracy"), (ax_loss, "loss", "Loss")]:
    ax.plot(history.history[metric], marker="o", label="Training")
    ax.plot(history.history[f"val_{metric}"], marker="o", label="Validation")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)
class_names = ["Negative", "Positive"]

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix (test set)")
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

## 8. Predict on new reviews

In [ ]:
def encode_review(text):
    """Encode raw text exactly like the training data: START token, word ids, OOV, same padding."""
    ids = [START]
    for word in re.findall(r"[a-z0-9']+", text.lower()):
        idx = word_index.get(word, -INDEX_FROM) + INDEX_FROM
        ids.append(idx if INDEX_FROM <= idx < VOCAB_SIZE else OOV)
    return pad_sequences([ids], maxlen=MAX_LEN, padding="post", truncating="pre")


def predict_sentiment(text):
    score = float(model.predict(encode_review(text), verbose=0)[0][0])
    label = "Positive" if score > 0.5 else "Negative"
    print(f"{label:8s} {score:.4f}  {text}")
    return label, score


examples = [
    "This movie is amazing and I loved it",
    "Worst movie ever, total waste of time",
    "The visual effects were stunning, but the plot was nonexistent.",
    "The acting was good but the story was total garbage",
    "This movie was not bad at all",                                            # negation
    "The movie was so long that I finished my graduation while watching it",   # sarcasm
]
for review in examples:
    predict_sentiment(review)

## 9. Save the model

In [ ]:
model.save("sentiment_model_bilstm.keras")
print("Saved: sentiment_model_bilstm.keras")